# Drift Simulation, Monitoring & Alerting

## Section 0 — Build reference data in-memory

In [0]:
%pip install databricks-feature-engineering evidently
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

fe = FeatureEngineeringClient()
V3 = "mlo.features.weather_daily_v3"

lookups = [FeatureLookup(table_name=V3, lookup_key=["station"], timestamp_lookup_key="date")]
ts = fe.create_training_set(
    df=spark.table("mlo.features.weather_labels"),
    feature_lookups=lookups,
    label="Bad",
)

reference_df = ts.load_df().toPandas().sort_values("date").reset_index(drop=True)
print(f"Loaded {len(reference_df)} rows, {len(reference_df.columns)} columns from {V3}")
reference_df[["station", "date"]].head()

## Section 1 — Reuse drift report logic from drift_monitoring.py

In [0]:
import sys, os

try:
    from drift_monitoring import get_feature_columns, run_drift_report, print_summary
except ImportError:
    repo_dir = os.path.dirname(os.path.abspath("__file__"))
    if repo_dir not in sys.path:
        sys.path.append(repo_dir)
    from drift_monitoring import get_feature_columns, run_drift_report, print_summary

feature_columns = get_feature_columns(reference_df)
print(f"Auto-detected {len(feature_columns)} feature columns")

## Section 2 — Corruption scenarios (n=500, fixes the sampling-noise issue at n=200)

In [0]:
import numpy as np

RANDOM_STATE = 1
SAMPLE_SIZE = 500  # was 200 -- several unrelated station columns sat right at the 0.1
                    # threshold from sampling noise alone, flagged as "moderate" in every
                    # scenario regardless of what was actually corrupted.


def corrupt_sensor_shift(df, feature_columns):
    out = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).copy()

    def perturb_matching(substring, fn, max_cols=1):
        matches = [c for c in feature_columns if substring in c][:max_cols]
        for c in matches:
            out[c] = fn(out[c])
        return matches

    touched = []
    touched += perturb_matching("TMAX", lambda s: s + 12)
    touched += perturb_matching("AWND", lambda s: s * 1.5)
    touched += perturb_matching("roll3", lambda s: s + 2.0)
    touched += perturb_matching("PRCP_lag", lambda s: s + 1.5)
    return out, f"Sensor/seasonal shift (n={SAMPLE_SIZE}): touched {touched}"


def corrupt_missing_data(df, feature_columns, frac=0.3, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    out = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=random_state).copy()
    for col in feature_columns:
        mask = rng.random(len(out)) < frac
        out.loc[mask, col] = np.nan
    return out, f"Missing data: ~{int(frac * 100)}% nulled per feature (n={SAMPLE_SIZE})"


def corrupt_schema_change(df, feature_columns):
    out = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).copy()
    rename_col, drop_col = feature_columns[0], feature_columns[-1]
    out = out.rename(columns={rename_col: f"{rename_col}_renamed"}).drop(columns=[drop_col])
    return out, f"Schema change: {rename_col} renamed, {drop_col} dropped (n={SAMPLE_SIZE})"


def corrupt_unit_error(df, feature_columns):
    out = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).copy()
    wind_cols = [c for c in feature_columns if "AWND" in c or "WSF" in c][:1]
    for c in wind_cols:
        out[c] = out[c] * 1.609
    return out, f"Unit error: {wind_cols} in km/h not converted (n={SAMPLE_SIZE})"


def corrupt_stuck_sensor(df, feature_columns):
    out = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).copy()
    temp_cols = [c for c in feature_columns if "TMIN" in c][:1] or feature_columns[:1]
    for c in temp_cols:
        if len(out) > 0:
            out[c] = out[c].iloc[0]
    return out, f"Stuck sensor: {temp_cols} frozen (n={SAMPLE_SIZE})"


def corrupt_outliers(df, feature_columns, frac=0.05, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    out = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=random_state).copy()
    temp_cols = [c for c in feature_columns if "TMAX" in c][:1] or feature_columns[:1]
    n_outliers = max(1, int(len(out) * frac))
    idx = rng.choice(out.index, size=n_outliers, replace=False)
    for c in temp_cols:
        out.loc[idx, c] = out.loc[idx, c] * rng.uniform(3, 5, size=n_outliers)
    return out, f"Outliers: {n_outliers} rows, {temp_cols} inflated 3-5x (n={SAMPLE_SIZE})"


SCENARIOS = {
    "sensor_shift": corrupt_sensor_shift, "missing_data": corrupt_missing_data,
    "schema_change": corrupt_schema_change, "unit_error": corrupt_unit_error,
    "stuck_sensor": corrupt_stuck_sensor, "outliers": corrupt_outliers,
}

## Section 3 — Run drift reports

In [0]:
corrupted_results = {}
drift_results = {}

for name, fn in SCENARIOS.items():
    corrupted_df, description = fn(reference_df, feature_columns)
    corrupted_results[name] = corrupted_df
    shared_cols = [c for c in feature_columns if c in corrupted_df.columns]

    result = run_drift_report(reference_df, corrupted_df, shared_cols)
    drift_results[name] = result
    print(f"\n[{name}] {description}")
    print_summary(result)

    import shutil
    shutil.move("drift_report.html", f"drift_report_{name}.html")

## Section 4 — Live endpoint test

In [0]:
import requests

ENDPOINT_URL = "https://dbc-6e95f6fb-0d49.cloud.databricks.com/serving-endpoints/endpoint_weather/invocations"
DATABRICKS_TOKEN = "TOKEN"

headers = {"Authorization": f"Bearer {DATABRICKS_TOKEN}", "Content-Type": "application/json"}


def send_to_endpoint(records, label, n_rows=5):
    sample = records[:n_rows]
    try:
        resp = requests.post(ENDPOINT_URL, headers=headers, json={"dataframe_records": sample}, timeout=60)
        return {"label": label, "status_code": resp.status_code, "ok": resp.ok,
                "response_preview": (resp.text[:500] if resp.text else "")}
    except requests.exceptions.RequestException as e:
        return {"label": label, "status_code": None, "ok": False,
                "response_preview": f"Request failed before reaching the endpoint: {e}"}


# Test 1: real station/date pairs the endpoint should be able to resolve
real_lookup_records = reference_df[["station", "date"]].head(5).copy()
real_lookup_records["date"] = real_lookup_records["date"].astype(str)
real_records = real_lookup_records.to_dict(orient="records")

result_real = send_to_endpoint(real_records, "real_station_date")
print(f"[real_station_date] status={result_real['status_code']} ok={result_real['ok']}")
print(f"  response: {result_real['response_preview']}")

# Test 2: a station/date the feature table can't resolve (simulated missing upstream data)
fake_records = [{"station": reference_df['station'].iloc[0], "date": "2030-01-01"}]
result_fake = send_to_endpoint(fake_records, "unresolvable_date")
print(f"\n[unresolvable_date] status={result_fake['status_code']} ok={result_fake['ok']}")
print(f"  response: {result_fake['response_preview']}")

## Section 5 — Automated evaluation & alerting

In [0]:
import pandas as pd


def classify_severity(value):
    if value >= 0.3:
        return "strong"
    elif value >= 0.1:
        return "moderate"
    return "none"


def summarize_drift_severity(result):
    payload = result.dict()
    counts = {"strong": 0, "moderate": 0, "none": 0}
    flagged = []
    for metric in payload.get("metrics", []):
        name = metric.get("metric_name", "")
        if not name.startswith("ValueDrift"):
            continue
        value = metric.get("value")
        if value is None:
            continue
        severity = classify_severity(value)
        counts[severity] += 1
        if severity != "none":
            flagged.append((name, round(value, 3), severity))
    return counts, flagged


def monitoring_decision(counts, scenario_name):
    strong, moderate = counts["strong"], counts["moderate"]
    if strong >= 3:
        decision = "Investigate immediately: multiple features show strong drift."
    elif strong >= 1:
        decision = "Investigate: at least one feature shows strong drift."
    elif moderate >= 3:
        decision = "Monitor closely: several features show moderate drift."
    else:
        decision = "No major drift action required based on current thresholds."
    return {"scenario": scenario_name, "strong_drift_count": strong,
            "moderate_drift_count": moderate, "decision": decision}


decisions = []
for name, result in drift_results.items():
    counts, flagged = summarize_drift_severity(result)
    decision = monitoring_decision(counts, name)
    decisions.append(decision)
    print(f"\n[{name}] strong={counts['strong']} moderate={counts['moderate']}")
    if flagged:
        print(f"  flagged: {flagged[:5]}{'...' if len(flagged) > 5 else ''}")
    print(f"  -> {decision['decision']}")

decisions_df = pd.DataFrame(decisions)
print("\n" + "=" * 70)
print("Automated monitoring decisions -- #24")
print("=" * 70)
print(decisions_df.to_string(index=False))

In [0]:
# Section 6 — Slide visuals

import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#888888",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

# Pull scenario order + decisions straight from Section 5's results
decisions_by_scenario = {d["scenario"]: d for d in decisions}
scenario_order = sorted(
    decisions_by_scenario,
    key=lambda s: (-decisions_by_scenario[s]["strong_drift_count"],
                   -decisions_by_scenario[s]["moderate_drift_count"])
)

strong = [decisions_by_scenario[s]["strong_drift_count"] for s in scenario_order]
moderate = [decisions_by_scenario[s]["moderate_drift_count"] for s in scenario_order]
decision_labels = [decisions_by_scenario[s]["decision"] for s in scenario_order]

# Pull the aggregate DriftedColumnsCount share from the actual Evidently results
drift_share_pct = []
for s in scenario_order:
    payload = drift_results[s].dict()
    share = 0.0
    for metric in payload.get("metrics", []):
        if metric.get("metric_name", "").startswith("DriftedColumnsCount"):
            share = metric.get("value", {}).get("share", 0.0) * 100
            break
    drift_share_pct.append(round(share, 1))


# ---- Chart 1: aggregate drift metric vs. its own alert threshold ----
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#c0392b" if "Investigate" in d else "#95a5a6" for d in decision_labels]
y_pos = range(len(scenario_order))

ax.barh(y_pos, drift_share_pct, color=colors, height=0.55)
ax.axvline(50, color="#2c3e50", linestyle="--", linewidth=1.5)
ax.text(51, len(scenario_order) - 0.3, "Evidently's alert\nthreshold (50%)",
        va="top", fontsize=11, color="#2c3e50")

ax.set_yticks(y_pos)
ax.set_yticklabels([s.replace("_", " ").title() for s in scenario_order], fontsize=13)
ax.set_xlabel("Share of columns flagged as drifted (%)", fontsize=12)
ax.set_xlim(0, max(60, max(drift_share_pct) + 15))
ax.invert_yaxis()

for i, v in enumerate(drift_share_pct):
    ax.text(v + 1, i, f"{v}%", va="center", fontsize=11)

plt.title("Aggregate drift metric never crosses its own alert threshold",
          fontsize=13, loc="left", pad=15)
plt.tight_layout()
plt.savefig("aggregate_drift_gap.png", dpi=200, bbox_inches="tight")
plt.show()


# ---- Chart 2: per-column severity -> decision (fixed labels + legend position) ----
fig, ax = plt.subplots(figsize=(9, 4.5))
y_pos = range(len(scenario_order))

ax.barh(y_pos, strong, color="#c0392b", height=0.5, label="Strong drift (columns)")
ax.barh(y_pos, moderate, left=strong, color="#e67e22", height=0.5, label="Moderate drift (columns)")

ax.set_yticks(y_pos)
ax.set_yticklabels([s.replace("_", " ").title() for s in scenario_order], fontsize=13)
ax.set_xlabel("Number of columns flagged", fontsize=12)
ax.set_xlim(0, max(strong[i] + moderate[i] for i in range(len(scenario_order))) + 2.5)
ax.invert_yaxis()
ax.legend(loc="lower right", frameon=False, fontsize=11)

# Short labels: map each decision to a fixed short phrase instead of
# splitting on ":" (which only some decisions contain)
short_labels = {
    "Investigate immediately: multiple features show strong drift.": "Investigate immediately",
    "Investigate: at least one feature shows strong drift.": "Investigate",
    "Monitor closely: several features show moderate drift.": "Monitor closely",
    "No major drift action required based on current thresholds.": "No action",
}

for i, (s, m, d) in enumerate(zip(strong, moderate, decision_labels)):
    label = short_labels.get(d, d)
    ax.text(s + m + 0.15, i, label, va="center", fontsize=10.5, color="#2c3e50")

plt.title("Per-column severity correctly differentiates scenario risk",
          fontsize=13, loc="left", pad=15)
plt.tight_layout()
plt.savefig("severity_decisions.png", dpi=200, bbox_inches="tight")
plt.show()


# ---- Table: live endpoint test results (#22 finding) ----
import pandas as pd

endpoint_table = pd.DataFrame([
    {"Test": "Real station/date", "Status": result_real["status_code"],
     "Response": result_real["response_preview"]},
    {"Test": "Unresolvable date (2030-01-01)", "Status": result_fake["status_code"],
     "Response": result_fake["response_preview"]},
])
print("\nLive endpoint test -- #22 finding:")
print(endpoint_table.to_string(index=False))